UNIFESP X-Ray — Final push: Ensemble + TTA + Threshold Tuning

Trains the 2 best Step 2 baselines (Swin-Tiny, DenseNet121) and both Step 3 hybrid variants (plain / hierarchical loss), then combines all 4 with:

Test-time augmentation (horizontal flip, averaged) -- free accuracy at inference
Ensembling (average probabilities across all 4 models) -- usually the biggest single gain
Per-class threshold tuning (instead of a flat 0.5 cutoff) -- squeezes out the imbalanced classes

Current best (single model): Swin-Tiny, sample_mean_f1 = 0.928 Real Kaggle competition 1st place: 0.934 (115 teams, private leaderboard) Goal of this notebook: close that ~0.006 gap without any new architecture -- just better use of what we've already built.

Fully self-contained -- only attach felipekitamura/unifesp-xray-bodypart-classification (Notebook Settings → Data) and turn on GPU T4 x2 (Notebook Settings → Accelerator).

1. Detect the attached images dataset

In [ ]:
import os

images_root = None
for cur, dirs, files in os.walk("/kaggle/input"):
    if "train" in dirs and "test" in dirs:
        images_root = cur
        dirs[:] = [d for d in dirs if d not in ("train", "test")]
        break

assert images_root is not None, (
    "Could not find a dataset with train/ and test/ folders under /kaggle/input. "
    "Attach felipekitamura/unifesp-xray-bodypart-classification (Notebook Settings -> Data)."
)
print("Images dataset:", images_root)
IMAGES_DATASET = images_root

Images dataset: /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification


2. Write the pipeline code (embedded, no external dataset needed)

In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/__init__.py", "w", encoding="utf-8") as f:
    f.write(r''''''.lstrip('\n'))
print("wrote src/__init__.py")

wrote src/__init__.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/config.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Central config. Two roots, because on Kaggle they'll usually be two
different attached datasets:
  - DATA_ROOT: where train/, test/ (the DICOM images) live
  - INDEX_ROOT: where index.csv and splits/folds.csv live

Locally both default to the repo root. On Kaggle, set them explicitly as
environment variables in the notebook's first cell, e.g.:
    import os
    os.environ["UNIFESP_DATA_ROOT"] = "/kaggle/input/unifesp-xray-bodypart-classification"
    os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/input/unifesp-index-files"  # your uploaded index.csv+splits/
    os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working"
(see KAGGLE_SETUP.md for the full first-cell snippet.)
"""
import os

_THIS_DIR = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(_THIS_DIR)

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    DATA_ROOT = os.environ.get("UNIFESP_DATA_ROOT", "/kaggle/input")
    INDEX_ROOT = os.environ.get("UNIFESP_INDEX_ROOT", DATA_ROOT)
    OUTPUT_DIR = os.environ.get("UNIFESP_OUTPUT_DIR", "/kaggle/working")
else:
    DATA_ROOT = os.environ.get("UNIFESP_DATA_ROOT", _REPO_ROOT)
    INDEX_ROOT = os.environ.get("UNIFESP_INDEX_ROOT", DATA_ROOT)
    OUTPUT_DIR = os.environ.get("UNIFESP_OUTPUT_DIR", os.path.join(_REPO_ROOT, "runs"))

INDEX_CSV = os.path.join(INDEX_ROOT, "index.csv")
FOLDS_CSV = os.path.join(INDEX_ROOT, "splits", "folds.csv")

# Optional: where cache_images.py pre-decodes DICOM -> PNG once, so every
# later training run reads a small PNG instead of re-decoding DICOM every
# epoch (top Kaggle finishers reported this cuts training from hours to
# minutes). None (default) means "no cache, decode DICOM every time" --
# fully backward compatible, nothing breaks if you skip this step.
CACHE_DIR = os.environ.get("UNIFESP_CACHE_DIR", os.path.join(OUTPUT_DIR, "image_cache") if ON_KAGGLE else None)

LABEL_NAMES = {
    0: "Abdomen", 1: "Ankle", 2: "Cervical Spine", 3: "Chest", 4: "Clavicles",
    5: "Elbow", 6: "Feet", 7: "Finger", 8: "Forearm", 9: "Hand", 10: "Hip",
    11: "Knee", 12: "Lower Leg", 13: "Lumbar Spine", 14: "Others", 15: "Pelvis",
    16: "Shoulder", 17: "Sinus", 18: "Skull", 19: "Thigh", 20: "Thoracic Spine",
    21: "Wrist",
}
N_CLASSES = len(LABEL_NAMES)
IMG_SIZE = 256
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
'''.lstrip('\n'))
print("wrote src/config.py")

wrote src/config.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/hierarchy.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Anatomical grouping of the 22 UNIFESP classes, used by the hierarchical loss
(losses_hierarchical.py). Grouping follows the same idea as Cheng et al.
2025/2026 (BMC Med Imaging) -- broad anatomical region as a coarser level
above the fine-grained label -- adapted to UNIFESP's 22 classes instead of
their 36 SNOMED codes.
"""
from . import config as C

GROUPS = {
    "Head": [17, 18],                       # Sinus, Skull
    "Spine": [2, 13, 20],                   # Cervical, Lumbar, Thoracic Spine
    "Trunk": [0, 3, 15],                     # Abdomen, Chest, Pelvis
    "UpperLimb": [4, 5, 7, 8, 9, 16, 21],    # Clavicles, Elbow, Finger, Forearm, Hand, Shoulder, Wrist
    "LowerLimb": [1, 6, 10, 11, 12, 19],     # Ankle, Feet, Hip, Knee, Lower Leg, Thigh
    "Other": [14],                          # Others
}

GROUP_NAMES = list(GROUPS.keys())
N_GROUPS = len(GROUP_NAMES)

# label_id (0..21) -> group index (0..5)
LABEL_TO_GROUP = {}
for g_idx, (g_name, label_ids) in enumerate(GROUPS.items()):
    for lid in label_ids:
        LABEL_TO_GROUP[lid] = g_idx

assert len(LABEL_TO_GROUP) == C.N_CLASSES, "every one of the 22 classes must map to exactly one group"


def build_label_to_group_matrix():
    """(N_CLASSES, N_GROUPS) 0/1 matrix: row k is a one-hot for class k's group.
    Multiplying a (batch, N_CLASSES) prediction/target by this gives the
    per-group aggregate (used by the hierarchical loss)."""
    import numpy as np
    m = np.zeros((C.N_CLASSES, N_GROUPS), dtype="float32")
    for lid, g_idx in LABEL_TO_GROUP.items():
        m[lid, g_idx] = 1.0
    return m
'''.lstrip('\n'))
print("wrote src/hierarchy.py")

wrote src/hierarchy.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/losses.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 2: plain multilabel loss with inverse-frequency class weighting
(dataset is heavily imbalanced: Chest=738 vs Clavicles=9 images).

Step 3 will add a hierarchical variant in losses_hierarchical.py that groups
labels into anatomical super-categories (spine, upper limb, lower limb, ...)
and penalizes cross-group errors more than within-group ones.
"""
import numpy as np
import torch
import torch.nn as nn


def compute_pos_weight(label_matrix, cap=20.0):
    """label_matrix: (N, n_classes) 0/1 array from the train split only."""
    n = label_matrix.shape[0]
    pos = label_matrix.sum(axis=0)
    pos = np.clip(pos, 1, None)  # avoid div-by-zero for classes with 0 positives in a fold
    weight = (n - pos) / pos
    weight = np.clip(weight, 1.0, cap)  # cap so the rarest class doesn't blow up the loss scale
    return torch.tensor(weight, dtype=torch.float32)


def build_loss(label_matrix=None, device="cpu"):
    pos_weight = compute_pos_weight(label_matrix) if label_matrix is not None else None
    if pos_weight is not None:
        pos_weight = pos_weight.to(device)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
'''.lstrip('\n'))
print("wrote src/losses.py")

wrote src/losses.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/losses_hierarchical.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 3 novelty component: hierarchical multilabel loss.

Plain BCE (Step 2) treats every misclassification equally -- confusing
Lumbar Spine for Thoracic Spine costs the same as confusing it for Hand.
That's not how these errors matter clinically or visually: adjacent/related
body parts get confused far more often (our EDA's top co-occurring pairs
were Lumbar+Thoracic Spine and Hand+Wrist), and a router that sends a spine
image to the wrong spine model is a much smaller failure than sending it to
a hand model.

We add a second loss term at the coarser anatomical-group level (see
hierarchy.py: Head/Spine/Trunk/UpperLimb/LowerLimb/Other) computed from the
same logits, so the model is pushed to at least get the broad region right
even when it's unsure of the exact class. This follows the same idea as
Cheng et al.'s hierarchical loss (BMC Med Imaging, 2025/2026) adapted to
UNIFESP's 22-class/6-group structure.
"""
import torch
import torch.nn as nn

from .hierarchy import N_GROUPS, build_label_to_group_matrix
from .losses import compute_pos_weight


class HierarchicalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, group_weight=0.3, device="cpu"):
        super().__init__()
        self.group_weight = group_weight
        self.fine_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        label_to_group = torch.tensor(build_label_to_group_matrix())  # (N_CLASSES, N_GROUPS)
        self.register_buffer("label_to_group", label_to_group)

    def forward(self, logits, targets):
        fine = self.fine_loss(logits, targets)

        # Group-level logit = max of the fine LOGITS belonging to that group
        # (not sigmoid probabilities -> plain BCELoss/binary_cross_entropy on
        # probabilities is explicitly unsafe under autocast/fp16, which is
        # what actually crashed on Kaggle: "torch.nn.functional.
        # binary_cross_entropy ... are unsafe to autocast". Staying in logit
        # space and using binary_cross_entropy_with_logits avoids that, and
        # max-of-logits is still a monotonic, differentiable "is any class in
        # this group present" signal.
        not_in_group = (self.label_to_group == 0).T            # (N_GROUPS, N_CLASSES) bool
        group_logit = torch.stack([
            logits.masked_fill(not_in_group[g], float("-inf")).max(dim=1).values
            for g in range(N_GROUPS)
        ], dim=1)                                               # (B, N_GROUPS)
        group_target = (targets @ self.label_to_group).clamp(max=1.0)  # (B, N_GROUPS)

        group = nn.functional.binary_cross_entropy_with_logits(group_logit, group_target)
        return fine + self.group_weight * group


def build_hierarchical_loss(label_matrix=None, group_weight=0.3, device="cpu"):
    pos_weight = compute_pos_weight(label_matrix) if label_matrix is not None else None
    if pos_weight is not None:
        pos_weight = pos_weight.to(device)
    return HierarchicalBCELoss(pos_weight=pos_weight, group_weight=group_weight).to(device)
'''.lstrip('\n'))
print("wrote src/losses_hierarchical.py")

wrote src/losses_hierarchical.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/dataset.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
UNIFESP DICOM dataset for multilabel body-part classification.

Handles the two DICOM quirks we found in EDA:
  - MONOCHROME1 vs MONOCHROME2 photometric interpretation (2367 vs 114 files)
    -> MONOCHROME1 is inverted, so we flip it to a consistent polarity.
  - Variable BitsStored (8/10/12/15-bit) -> min-max normalize per image instead
    of assuming a fixed bit depth.
"""
import os
import numpy as np
import pydicom
import torch
from PIL import Image
from torch.utils.data import Dataset

from . import config as C


def load_cached_or_decode(uid, dcm_path, cache_dir):
    """Read uid's pre-cached PNG if cache_dir has one; otherwise decode the
    DICOM and, if cache_dir is set, write the PNG for next time. Write is
    atomic (temp file + os.replace) so two training processes caching the
    same image concurrently (e.g. two backbones on two GPUs both hitting
    fold 0's validation set) never leaves a half-written file for the other
    to read -- worst case both just redundantly decode it once.
    """
    if cache_dir is not None:
        png_path = os.path.join(cache_dir, f"{uid}.png")
        if os.path.exists(png_path):
            return np.array(Image.open(png_path).convert("RGB"))

    image = load_dicom_as_image(dcm_path)

    if cache_dir is not None:
        os.makedirs(cache_dir, exist_ok=True)
        tmp_path = os.path.join(cache_dir, f".{uid}.{os.getpid()}.tmp.png")
        Image.fromarray(image).save(tmp_path)
        os.replace(tmp_path, os.path.join(cache_dir, f"{uid}.png"))

    return image


def load_dicom_as_image(path):
    ds = pydicom.dcmread(path, force=True)
    arr = ds.pixel_array.astype(np.float32)

    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    arr = arr * slope + intercept

    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        arr = arr.max() - arr

    lo, hi = arr.min(), arr.max()
    arr = (arr - lo) / (hi - lo + 1e-6)
    arr = np.stack([arr, arr, arr], axis=-1)  # 1-channel X-ray -> pseudo-RGB for ImageNet backbones
    return (arr * 255.0).astype(np.uint8)


def _basename_uid(path):
    b = os.path.basename(path)[:-4]  # strip .dcm
    if b.endswith("-c"):
        b = b[:-2]
    return b


def build_uid_to_path(data_root):
    """Walk data_root once and map uid -> absolute file path. We do this at
    runtime instead of trusting a stored relative path, because the raw
    DICOM folders are nested inconsistently (a local zip-extract quirk gave
    us train/train/train/<id>/..., while the Kaggle-mounted dataset or a
    re-extraction elsewhere may only have train/<id>/...). Matching by the
    DICOM filename's UID instead of a fixed path makes this robust to
    wherever the raw files actually end up on disk."""
    lookup = {}
    for root, _dirs, files in os.walk(data_root):
        for f in files:
            if f.lower().endswith(".dcm"):
                lookup[_basename_uid(f)] = os.path.join(root, f)
    return lookup


class UnifespDataset(Dataset):
    def __init__(self, df, data_root, transform=None, has_labels=True, uid_to_path=None, cache_dir=C.CACHE_DIR):
        """
        df: a slice of index.csv (optionally merged with folds.csv), must have a 'uid' column
        data_root: base dir containing the (possibly nested) train/test DICOM folders
        uid_to_path: pass a precomputed dict to avoid re-walking data_root for every split
        cache_dir: optional PNG cache (see load_cached_or_decode); None disables caching
        """
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.transform = transform
        self.has_labels = has_labels
        self.label_cols = [f"label_{k}_{C.LABEL_NAMES[k].replace(' ', '_')}" for k in range(C.N_CLASSES)]
        self.uid_to_path = uid_to_path if uid_to_path is not None else build_uid_to_path(data_root)
        self.cache_dir = cache_dir

        missing = set(self.df["uid"]) - set(self.uid_to_path)
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} uid(s) in this split have no matching .dcm file under {data_root} "
                f"(e.g. {next(iter(missing))}). Check UNIFESP_DATA_ROOT points at the right dataset."
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row["uid"]
        path = self.uid_to_path[uid]
        image = load_cached_or_decode(uid, path, self.cache_dir)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.has_labels:
            target = row[self.label_cols].to_numpy(dtype=np.float32)
            target = torch.from_numpy(target)
            return image, target
        return image, row["uid"]


def get_transforms(img_size=C.IMG_SIZE, train=True):
    """Train-time augmentation.

    We tried matching the UNIFESP Kaggle competition's top finishers
    (2nd place writeup used GaussNoise + GaussianBlur + ShiftScaleRotate +
    brightness/contrast) and measured it head-to-head on our own fold 0:

        backbone          light-aug  heavy-aug(+noise/blur)
        efficientnetv2_s     0.876      0.851   (worse)
        efficientnet_b3      0.866      0.847   (worse)
        inception_v3         0.872      0.841   (worse)
        convnext_tiny        0.738      0.558   (much worse)
        densenet121          0.869      0.873   (~same)
        vgg16                0.827      0.836   (~same)

    GaussNoise/GaussianBlur made 6/8 backbones worse (macro_f1), ConvNeXt-
    Tiny catastrophically so -- likely because blurring removes the fine
    bone-edge texture that separates visually similar classes (Lumbar vs
    Thoracic Spine, Hand vs Wrist), and our fixed 30-epoch budget (no
    warmup, no early stopping) doesn't give heavier augmentation enough
    iterations to pay off the way it did for the competition's longer,
    early-stopped fastai runs. So: geometric augmentation (flip, mild
    shift/scale/rotate) + brightness/contrast, no synthetic noise/blur.
    This negative result is itself worth keeping for the paper's ablation
    section.
    """
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])
'''.lstrip('\n'))
print("wrote src/dataset.py")

wrote src/dataset.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/engine.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Train/validate loop with checkpoint + auto-resume, built around Kaggle's
12-hour session limit: pass --max_hours (default 11.0, leaving a ~1h buffer).
If the wall-clock budget runs out, we save a checkpoint after the current
epoch finishes and exit cleanly (exit code 0) instead of getting killed
mid-epoch by Kaggle. Re-running the same command resumes from epoch+1.
"""
import os
import time
import json
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score, cohen_kappa_score


def _safe_macro_auc(y_true, y_prob):
    """Macro AUC skipping classes with 0 or all-positive ground truth in this split
    (undefined otherwise) -- common for the rarest UNIFESP classes in a single fold."""
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.min() == col.max():
            continue
        aucs.append(roc_auc_score(col, y_prob[:, c]))
    return float(np.mean(aucs)) if aucs else float("nan")


def sample_mean_f1(y_true, y_pred):
    """Per-image F1 averaged over images -- this is the UNIFESP Kaggle
    competition's actual leaderboard metric ("Mean F1-Score": one F1 per
    SOPInstanceUID between predicted and true label sets, then averaged).
    It's NOT the same as macro_f1 (averaged per-CLASS): since ~90% of images
    are single-label and Chest dominates, a model that nails the common
    classes scores high here even if it's weak on rare classes -- which is
    exactly why the real Kaggle leaderboard (top score 0.934, kaggle.com/
    competitions/unifesp-x-ray-body-part-classifier) sits well above the
    86.94% "accuracy" reported in Krishnan & Padmavathi 2025 (Procedia CS) --
    different metric, not actually a stronger model. This function is what
    makes our numbers comparable to that leaderboard.
    """
    f1s = []
    for i in range(y_true.shape[0]):
        t, p = y_true[i], y_pred[i]
        tp = np.sum((t == 1) & (p == 1))
        fp = np.sum((t == 0) & (p == 1))
        fn = np.sum((t == 1) & (p == 0))
        if tp == 0 and fp == 0 and fn == 0:
            f1s.append(1.0)  # both empty -- shouldn't happen here (every image has >=1 label) but defined for safety
            continue
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))


def compute_metrics(y_true, y_prob, threshold=0.5):
    # Every value here is cast to a plain Python float (sklearn/numpy return
    # numpy scalars) -- these end up pickled inside checkpoint_*.pt via the
    # "history" list, and PyTorch >=2.6's default torch.load(weights_only=True)
    # refuses to unpickle numpy scalar types, which crashed evaluate_checkpoints.py
    # with "Unsupported global: GLOBAL numpy._core.multiarray.scalar".
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "sample_mean_f1": float(sample_mean_f1(y_true, y_pred)),  # <- compare this one to the Kaggle leaderboard
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_auc": float(_safe_macro_auc(y_true, y_prob)),
        "subset_accuracy": float((y_pred == y_true).all(axis=1).mean()),
        "kappa_flat": float(cohen_kappa_score(y_true.flatten(), y_pred.flatten())),
    }


def train_one_epoch(model, loader, optimizer, loss_fn, device, scaler=None):
    model.train()
    total_loss = 0.0
    n = 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                loss = loss_fn(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = loss_fn(logits, targets)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
    return total_loss / n


@torch.no_grad()
def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_probs, all_targets = [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = loss_fn(logits, targets)
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(targets.cpu().numpy())
    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    metrics = compute_metrics(y_true, y_prob)
    metrics["loss"] = total_loss / n
    return metrics, y_prob, y_true


def save_checkpoint(path, epoch, model, optimizer, scheduler, best_metric, history):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_metric": best_metric,
        "history": history,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)  # our own checkpoint, trusted
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt["scheduler_state"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    return ckpt["epoch"], ckpt["best_metric"], ckpt["history"]


def run_training(model, train_loader, val_loader, optimizer, scheduler, loss_fn,
                  device, run_dir, n_epochs=30, max_hours=11.0, metric_name="macro_f1"):
    os.makedirs(run_dir, exist_ok=True)
    last_ckpt = os.path.join(run_dir, "checkpoint_last.pt")
    best_ckpt = os.path.join(run_dir, "checkpoint_best.pt")
    history_path = os.path.join(run_dir, "history.json")

    start_epoch = 0
    best_metric = -float("inf")
    history = []
    if os.path.exists(last_ckpt):
        start_epoch, best_metric, history = load_checkpoint(last_ckpt, model, optimizer, scheduler, device)
        start_epoch += 1
        print(f"[resume] loaded checkpoint, continuing from epoch {start_epoch}/{n_epochs}")

    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
    run_start = time.time()
    budget_seconds = max_hours * 3600

    for epoch in range(start_epoch, n_epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device, scaler)
        val_metrics, _, _ = validate(model, val_loader, loss_fn, device)
        if scheduler is not None:
            scheduler.step()
        dt = time.time() - t0

        row = {"epoch": epoch, "train_loss": train_loss, "epoch_seconds": dt, **val_metrics}
        history.append(row)
        print(f"epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_metrics['loss']:.4f} "
              f"{metric_name}={val_metrics[metric_name]:.4f} ({dt:.1f}s)")

        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        if val_metrics[metric_name] > best_metric:
            best_metric = val_metrics[metric_name]
            save_checkpoint(best_ckpt, epoch, model, optimizer, scheduler, best_metric, history)
            print(f"  -> new best {metric_name}={best_metric:.4f}, saved checkpoint_best.pt")

        save_checkpoint(last_ckpt, epoch, model, optimizer, scheduler, best_metric, history)

        elapsed = time.time() - run_start
        if elapsed > budget_seconds:
            print(f"[time budget] {elapsed/3600:.2f}h elapsed > {max_hours}h limit, "
                  f"stopping after epoch {epoch}. Re-run the same command to resume.")
            break

    return history, best_metric
'''.lstrip('\n'))
print("wrote src/engine.py")

wrote src/engine.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/models.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Model factory.

Step 2 baselines (reproducing Krishnan & Padmavathi 2025's comparison, plus
modern backbones): vgg16, inception_v3, resnet50, densenet121, efficientnet_b3,
efficientnetv2_s, convnext_tiny, swin_tiny.

Step 3 novelty: HybridCNNTransformer (see models_hybrid.py) is registered
separately so baseline runs stay untouched while that's developed.
"""
import timm
import torch.nn as nn

from . import config as C

BASELINE_BACKBONES = {
    "vgg16": "vgg16",
    "inception_v3": "inception_v3",
    "resnet50": "resnet50",
    "densenet121": "densenet121",
    "efficientnet_b3": "efficientnet_b3",
    "efficientnetv2_s": "tf_efficientnetv2_s",
    "convnext_tiny": "convnext_tiny",
    "swin_tiny": "swin_tiny_patch4_window7_224",
}


def build_model(name, n_classes=C.N_CLASSES, pretrained=True):
    if name not in BASELINE_BACKBONES:
        raise ValueError(f"Unknown backbone '{name}'. Options: {list(BASELINE_BACKBONES)}")
    timm_name = BASELINE_BACKBONES[name]
    model = timm.create_model(timm_name, pretrained=pretrained, num_classes=n_classes)
    return model


def input_size_for(name):
    # swin_tiny_patch4_window7_224 is fixed at 224; everything else we train at C.IMG_SIZE (256)
    if name == "swin_tiny":
        return 224
    return C.IMG_SIZE
'''.lstrip('\n'))
print("wrote src/models.py")

wrote src/models.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/models_hybrid.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 3 novelty component: Hybrid CNN-Transformer with attention-based fusion.

Step 2's two best backbones were complementary: DenseNet121 (best macro_f1
0.875 -- strong on rare classes) and Swin-Tiny (best sample_mean_f1 0.928 --
strong on the common ones). The idea, following the dual-path CNN+Transformer
fusion pattern reported for musculoskeletal radiographs (Shah et al. 2026,
J. Imaging, Xception-Swin): the CNN branch's local conv features (bone edges,
cortical texture) and the Transformer branch's global shifted-window
attention (whole-image context) carry different information, and a learned
gate lets the model weight each per-sample instead of fixing their
contribution.

No prior UNIFESP work (the Krishnan 2025 paper, or any Kaggle competition
solution we found) uses a CNN+Transformer hybrid -- the competition's top
entries used single CNNs (ResNet50/152). This is new for this dataset.
"""
import timm
import torch
import torch.nn as nn

from . import config as C

HYBRID_INPUT_SIZE = 224  # swin_tiny_patch4_window7_224 needs 224; densenet121 is resolution-flexible


class HybridCNNTransformer(nn.Module):
    def __init__(self, n_classes=C.N_CLASSES, cnn_name="densenet121",
                 transformer_name="swin_tiny_patch4_window7_224", fusion_dim=512, pretrained=True):
        super().__init__()
        self.cnn = timm.create_model(cnn_name, pretrained=pretrained, num_classes=0)       # -> pooled feature vector
        self.transformer = timm.create_model(transformer_name, pretrained=pretrained, num_classes=0)

        cnn_dim = self.cnn.num_features
        trans_dim = self.transformer.num_features

        self.cnn_proj = nn.Linear(cnn_dim, fusion_dim)
        self.trans_proj = nn.Linear(trans_dim, fusion_dim)

        # learned per-sample gate over the two branches, from their concatenated raw features
        self.gate = nn.Sequential(
            nn.Linear(cnn_dim + trans_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(fusion_dim, n_classes),
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)                  # (B, cnn_dim)
        trans_feat = self.transformer(x)         # (B, trans_dim)

        gate_logits = self.gate(torch.cat([cnn_feat, trans_feat], dim=1))
        gate_weights = torch.softmax(gate_logits, dim=1)  # (B, 2) -- how much to trust CNN vs Transformer

        fused = (gate_weights[:, 0:1] * self.cnn_proj(cnn_feat)
                 + gate_weights[:, 1:2] * self.trans_proj(trans_feat))

        return self.classifier(fused)

    def forward_with_gate(self, x):
        """Same as forward, but also returns the per-sample gate weights --
        useful for Step 5 (explainability): shows whether the model is
        leaning on local texture (CNN) or global context (Transformer) for
        a given image, per class confusion pair."""
        cnn_feat = self.cnn(x)
        trans_feat = self.transformer(x)
        gate_logits = self.gate(torch.cat([cnn_feat, trans_feat], dim=1))
        gate_weights = torch.softmax(gate_logits, dim=1)
        fused = (gate_weights[:, 0:1] * self.cnn_proj(cnn_feat)
                 + gate_weights[:, 1:2] * self.trans_proj(trans_feat))
        return self.classifier(fused), gate_weights
'''.lstrip('\n'))
print("wrote src/models_hybrid.py")

wrote src/models_hybrid.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/build_index.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 1a: Build a flat index of the UNIFESP dataset.

Reads every DICOM's header (no pixel decode, so it's fast) and writes a
single flat CSV that the rest of the pipeline uses. Uses config.DATA_ROOT /
config.INDEX_ROOT (not a path relative to this file) so it works no matter
where the raw images or the output location actually are -- e.g. run fresh
inside a Kaggle notebook against the attached images dataset, writing to
/kaggle/working, with no separately-uploaded "index" dataset needed at all.

Output: <INDEX_ROOT>/index.csv with columns:
    uid, split, patient_id, study_uid, series_uid, labels, label_0_Abdomen, ...
(no filepath column -- dataset.py locates files by uid via build_uid_to_path,
which is robust to whatever nesting the raw folders happen to have.)
"""
import os
import pydicom
import pandas as pd

from . import config as C
from .dataset import build_uid_to_path

LABEL_NAMES = C.LABEL_NAMES
N_CLASSES = C.N_CLASSES


def main():
    print("Indexing DICOM files under", C.DATA_ROOT, "...")
    train_uid_to_path = build_uid_to_path(os.path.join(C.DATA_ROOT, "train"))
    test_uid_to_path = build_uid_to_path(os.path.join(C.DATA_ROOT, "test"))
    print(f"train dcm files: {len(train_uid_to_path)} | test dcm files: {len(test_uid_to_path)}")

    rows = []
    for split, uid_to_path in (("train", train_uid_to_path), ("test", test_uid_to_path)):
        for uid, path in uid_to_path.items():
            try:
                ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
                patient_id = str(getattr(ds, "PatientID", ""))
                study_uid = str(getattr(ds, "StudyInstanceUID", ""))
                series_uid = str(getattr(ds, "SeriesInstanceUID", ""))
            except Exception as e:
                patient_id = study_uid = series_uid = ""
                print("WARN: failed to read header for", path, e)
            rows.append({
                "uid": uid, "split": split,
                "patient_id": patient_id, "study_uid": study_uid, "series_uid": series_uid,
            })

    index_df = pd.DataFrame(rows)

    # attach multilabel targets for the train split
    train_csv = pd.read_csv(os.path.join(C.DATA_ROOT, "train.csv"))
    train_csv["SOPInstanceUID"] = train_csv["SOPInstanceUID"].str.strip()
    label_map = dict(zip(train_csv["SOPInstanceUID"], train_csv["Target"]))
    index_df["labels"] = index_df["uid"].map(label_map)

    # one-hot columns for the 22 classes (NaN/absent for test rows)
    for k in range(N_CLASSES):
        col = f"label_{k}_{LABEL_NAMES[k].replace(' ', '_')}"
        index_df[col] = index_df["labels"].apply(
            lambda s, k=k: int(k in [int(x) for x in str(s).split()]) if pd.notna(s) else pd.NA
        )

    os.makedirs(C.INDEX_ROOT, exist_ok=True)
    out_path = os.path.join(C.INDEX_ROOT, "index.csv")
    index_df.to_csv(out_path, index=False)
    print("Saved index to", out_path, "| rows:", len(index_df))

    # ---- patient-level leakage sanity check ----
    train_idx = index_df[index_df["split"] == "train"]
    test_idx = index_df[index_df["split"] == "test"]
    train_patients = set(train_idx["patient_id"]) - {""}
    test_patients = set(test_idx["patient_id"]) - {""}
    overlap = train_patients & test_patients
    print(f"\nUnique patients -> train: {len(train_patients)}, test: {len(test_patients)}")
    print(f"Patient overlap between train/test (Kaggle's own split): {len(overlap)}")

    dup = train_idx.groupby("patient_id").size()
    multi_image_patients = (dup > 1).sum()
    print(f"Train patients with >1 image: {multi_image_patients} / {len(dup)}")


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/build_index.py")

wrote src/build_index.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/make_splits.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 1b: Patient-level, multilabel-stratified K-fold split.

Why patient-level: build_index.py found 483/768 train patients have >1 image,
so a plain random (or even plain multilabel-stratified) image-level split can
put the same patient's images in both train and validation -> leaky, overly
optimistic validation scores. We group by patient_id first, then stratify.

Why multilabel-stratified: UNIFESP's 22 classes are heavily imbalanced
(Chest 738 vs Clavicles 9), so a plain GroupKFold can starve a fold of a rare
class entirely. We stratify on each patient's aggregated (union) label
vector using iterative stratification.

Output: splits/fold_{i}.csv with columns [uid, fold] for every train image.
"""
import os
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

from . import config as C

N_CLASSES = C.N_CLASSES
N_FOLDS = 5
SEED = C.SEED


def main():
    index_df = pd.read_csv(os.path.join(C.INDEX_ROOT, "index.csv"))
    train_df = index_df[index_df["split"] == "train"].copy()

    label_cols = [c for c in train_df.columns if c.startswith("label_")]
    assert len(label_cols) == N_CLASSES, label_cols

    # aggregate to patient level: union of all labels this patient's images carry
    patient_labels = (
        train_df.groupby("patient_id")[label_cols]
        .max()
        .reset_index()
    )
    patients = patient_labels["patient_id"].values
    Y = patient_labels[label_cols].values.astype(int)

    print(f"Patients: {len(patients)} | images: {len(train_df)}")
    print("Per-class positive patient count:\n", pd.Series(Y.sum(axis=0), index=label_cols))

    mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    patient_fold = np.full(len(patients), -1, dtype=int)
    for fold, (_, val_idx) in enumerate(mskf.split(patients, Y)):
        patient_fold[val_idx] = fold

    patient_to_fold = dict(zip(patients, patient_fold))
    train_df["fold"] = train_df["patient_id"].map(patient_to_fold)

    assert (train_df["fold"] >= 0).all(), "every image must get a fold"

    out_cols = ["uid", "patient_id", "fold"] + label_cols
    splits_dir = os.path.join(C.INDEX_ROOT, "splits")
    os.makedirs(splits_dir, exist_ok=True)
    out_path = os.path.join(splits_dir, "folds.csv")
    train_df[out_cols].to_csv(out_path, index=False)
    print("Saved", out_path)

    # sanity: no patient split across folds, per-fold class coverage
    leak = train_df.groupby("patient_id")["fold"].nunique()
    print("Patients appearing in >1 fold (should be 0):", (leak > 1).sum())

    print("\nPer-fold image count:\n", train_df["fold"].value_counts().sort_index())
    print("\nPer-fold positive image count per class:")
    print(train_df.groupby("fold")[label_cols].sum())


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/make_splits.py")

wrote src/make_splits.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/cache_images.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Optional one-time speed-up: pre-decode every DICOM into a PNG cache before
launching the 8 parallel training runs. UnifespDataset already caches
lazily on first read (see dataset.py's load_cached_or_decode), so this
script isn't required for correctness -- it just avoids every one of the
first 2 (parallel) backbones redundantly decoding the same ~1700 training
images during their first epoch. Top Kaggle finishers on this exact
competition reported DICOM decoding as their biggest bottleneck.

Kaggle usage, right after environment setup, before Step 2 training:
    python -m src.cache_images
"""
import time

from . import config as C
from .dataset import build_uid_to_path, load_cached_or_decode


def main():
    if C.CACHE_DIR is None:
        print("UNIFESP_CACHE_DIR / CACHE_DIR is not set -- nothing to do. "
              "Caching is off by default outside Kaggle; set the env var if you want it locally.")
        return

    print("Indexing DICOM files under", C.DATA_ROOT, "...")
    uid_to_path = build_uid_to_path(C.DATA_ROOT)
    print(f"Found {len(uid_to_path)} files. Caching to {C.CACHE_DIR} ...")

    t0 = time.time()
    for i, (uid, path) in enumerate(uid_to_path.items()):
        load_cached_or_decode(uid, path, C.CACHE_DIR)
        if (i + 1) % 250 == 0:
            print(f"  {i + 1}/{len(uid_to_path)} ({time.time() - t0:.1f}s elapsed)")

    print(f"Done. Cached {len(uid_to_path)} images in {time.time() - t0:.1f}s -> {C.CACHE_DIR}")


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/cache_images.py")

wrote src/cache_images.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/train.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 2 entry point: train one backbone on one fold.

Kaggle usage (12h session limit, dual GPU):
    python -m src.train --backbone efficientnet_b3 --fold 0 --epochs 30 --max_hours 11
If the session gets cut off or you stop it, just re-run the exact same
command later (new Kaggle session) -- it auto-resumes from checkpoint_last.pt
in OUTPUT_DIR/<run_name>/.

Local smoke test (CPU, tiny subset, no real training):
    python -m src.train --backbone resnet50 --fold 0 --epochs 1 --max_hours 0.01 --smoke_test
"""
import argparse
import os
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from . import config as C
from .dataset import UnifespDataset, get_transforms, build_uid_to_path
from .models import build_model, input_size_for
from .losses import build_loss
from .engine import run_training


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_dataframes(fold, smoke_test=False):
    index_df = pd.read_csv(C.INDEX_CSV)
    folds_df = pd.read_csv(C.FOLDS_CSV)

    train_index = index_df[index_df["split"] == "train"].merge(
        folds_df[["uid", "fold"]], on="uid", how="inner"
    )
    train_df = train_index[train_index["fold"] != fold].reset_index(drop=True)
    val_df = train_index[train_index["fold"] == fold].reset_index(drop=True)

    if smoke_test:
        train_df = train_df.iloc[:8].reset_index(drop=True)
        val_df = val_df.iloc[:4].reset_index(drop=True)
    return train_df, val_df


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--backbone", default="efficientnet_b3")
    p.add_argument("--fold", type=int, default=0)
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--max_hours", type=float, default=11.0)
    p.add_argument("--num_workers", type=int, default=4)
    p.add_argument("--seed", type=int, default=C.SEED)
    p.add_argument("--smoke_test", action="store_true")
    args = p.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.device_count() > 1:
        print(f"[info] {torch.cuda.device_count()} GPUs visible — this script uses GPU 0 only "
              f"by default; run separate backbones on separate GPUs in parallel for Step 2's "
              f"model comparison instead of DataParallel (simpler, and safer for the 12h budget).")

    train_df, val_df = build_dataframes(args.fold, smoke_test=args.smoke_test)
    print(f"train: {len(train_df)} images | val: {len(val_df)} images")

    print("indexing DICOM files under", C.DATA_ROOT, "...")
    uid_to_path = build_uid_to_path(C.DATA_ROOT)
    print(f"found {len(uid_to_path)} .dcm files")

    img_size = input_size_for(args.backbone)
    train_ds = UnifespDataset(train_df, C.DATA_ROOT, transform=get_transforms(img_size, train=True),
                               uid_to_path=uid_to_path)
    val_ds = UnifespDataset(val_df, C.DATA_ROOT, transform=get_transforms(img_size, train=False),
                             uid_to_path=uid_to_path)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                               num_workers=args.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                             num_workers=args.num_workers, pin_memory=True)

    model = build_model(args.backbone, pretrained=not args.smoke_test).to(device)

    label_cols = [f"label_{k}_{C.LABEL_NAMES[k].replace(' ', '_')}" for k in range(C.N_CLASSES)]
    loss_fn = build_loss(train_df[label_cols].to_numpy(dtype="float32"), device=device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    run_name = f"{args.backbone}_fold{args.fold}"
    run_dir = os.path.join(C.OUTPUT_DIR, run_name)

    history, best_metric = run_training(
        model, train_loader, val_loader, optimizer, scheduler, loss_fn,
        device, run_dir, n_epochs=args.epochs, max_hours=args.max_hours,
    )
    print(f"done. best macro_f1={best_metric:.4f}. checkpoints in {run_dir}")


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/train.py")

wrote src/train.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/train_hybrid.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 3 entry point: train the Hybrid CNN-Transformer, with a --loss flag to
run the plain-BCE vs hierarchical-loss ablation (same engine/checkpointing
as Step 2's train.py, so 12h-session auto-resume works the same way).

Kaggle usage (run both variants, one per GPU, for the ablation):
    CUDA_VISIBLE_DEVICES=0 python -m src.train_hybrid --loss plain --epochs 30 --max_hours 5
    CUDA_VISIBLE_DEVICES=1 python -m src.train_hybrid --loss hierarchical --epochs 30 --max_hours 5

Local smoke test (CPU, tiny subset):
    python -m src.train_hybrid --loss hierarchical --epochs 1 --max_hours 0.01 --smoke_test
"""
import argparse
import os
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from . import config as C
from .dataset import UnifespDataset, get_transforms, build_uid_to_path
from .models_hybrid import HybridCNNTransformer, HYBRID_INPUT_SIZE
from .losses import build_loss
from .losses_hierarchical import build_hierarchical_loss
from .engine import run_training


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_dataframes(fold, smoke_test=False):
    index_df = pd.read_csv(C.INDEX_CSV)
    folds_df = pd.read_csv(C.FOLDS_CSV)

    train_index = index_df[index_df["split"] == "train"].merge(
        folds_df[["uid", "fold"]], on="uid", how="inner"
    )
    train_df = train_index[train_index["fold"] != fold].reset_index(drop=True)
    val_df = train_index[train_index["fold"] == fold].reset_index(drop=True)

    if smoke_test:
        train_df = train_df.iloc[:8].reset_index(drop=True)
        val_df = val_df.iloc[:4].reset_index(drop=True)
    return train_df, val_df


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--loss", choices=["plain", "hierarchical"], default="hierarchical")
    p.add_argument("--fold", type=int, default=0)
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch_size", type=int, default=24)  # hybrid model is heavier than a single backbone
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--group_weight", type=float, default=0.3, help="hierarchical loss's group-level term weight")
    p.add_argument("--max_hours", type=float, default=11.0)
    p.add_argument("--num_workers", type=int, default=4)
    p.add_argument("--seed", type=int, default=C.SEED)
    p.add_argument("--smoke_test", action="store_true")
    args = p.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_df, val_df = build_dataframes(args.fold, smoke_test=args.smoke_test)
    print(f"train: {len(train_df)} images | val: {len(val_df)} images | loss: {args.loss}")

    print("indexing DICOM files under", C.DATA_ROOT, "...")
    uid_to_path = build_uid_to_path(C.DATA_ROOT)
    print(f"found {len(uid_to_path)} .dcm files")

    train_ds = UnifespDataset(train_df, C.DATA_ROOT, transform=get_transforms(HYBRID_INPUT_SIZE, train=True),
                               uid_to_path=uid_to_path)
    val_ds = UnifespDataset(val_df, C.DATA_ROOT, transform=get_transforms(HYBRID_INPUT_SIZE, train=False),
                             uid_to_path=uid_to_path)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                               num_workers=args.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                             num_workers=args.num_workers, pin_memory=True)

    model = HybridCNNTransformer(pretrained=not args.smoke_test).to(device)

    label_cols = [f"label_{k}_{C.LABEL_NAMES[k].replace(' ', '_')}" for k in range(C.N_CLASSES)]
    label_matrix = train_df[label_cols].to_numpy(dtype="float32")
    if args.loss == "hierarchical":
        loss_fn = build_hierarchical_loss(label_matrix, group_weight=args.group_weight, device=device)
    else:
        loss_fn = build_loss(label_matrix, device=device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    run_name = f"hybrid_{args.loss}_fold{args.fold}"
    run_dir = os.path.join(C.OUTPUT_DIR, run_name)

    history, best_metric = run_training(
        model, train_loader, val_loader, optimizer, scheduler, loss_fn,
        device, run_dir, n_epochs=args.epochs, max_hours=args.max_hours,
    )
    print(f"done. best macro_f1={best_metric:.4f}. checkpoints in {run_dir}")


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/train_hybrid.py")

wrote src/train_hybrid.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/evaluate_hybrid.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Step 3 evaluation: recompute metrics for the two hybrid-model checkpoints
(plain vs hierarchical loss) and print them next to the Step 2 baseline
leaderboard-comparable numbers, so the ablation and the "did Step 3 beat
Step 2" question are both answered in one place.

Kaggle usage (after both hybrid variants are trained):
    python -m src.evaluate_hybrid --fold 0
"""
import argparse
import os

import pandas as pd
import torch
from torch.utils.data import DataLoader

from . import config as C
from .dataset import UnifespDataset, get_transforms, build_uid_to_path
from .models_hybrid import HybridCNNTransformer, HYBRID_INPUT_SIZE
from .losses import build_loss
from .engine import validate


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--fold", type=int, default=0)
    p.add_argument("--batch_size", type=int, default=24)
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    index_df = pd.read_csv(C.INDEX_CSV)
    folds_df = pd.read_csv(C.FOLDS_CSV)
    train_index = index_df[index_df["split"] == "train"].merge(folds_df[["uid", "fold"]], on="uid", how="inner")
    val_df = train_index[train_index["fold"] == args.fold].reset_index(drop=True)

    print("indexing DICOM files under", C.DATA_ROOT, "...")
    uid_to_path = build_uid_to_path(C.DATA_ROOT)
    val_ds = UnifespDataset(val_df, C.DATA_ROOT, transform=get_transforms(HYBRID_INPUT_SIZE, train=False),
                             uid_to_path=uid_to_path)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    loss_fn = build_loss(None, device=device)

    rows = []
    for variant in ["plain", "hierarchical"]:
        run_dir = os.path.join(C.OUTPUT_DIR, f"hybrid_{variant}_fold{args.fold}")
        ckpt_path = os.path.join(run_dir, "checkpoint_best.pt")
        if not os.path.exists(ckpt_path):
            print(f"[skip] hybrid_{variant}: no checkpoint_best.pt in {run_dir}")
            continue

        model = HybridCNNTransformer(pretrained=False).to(device)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state"])

        metrics, _, _ = validate(model, val_loader, loss_fn, device)
        metrics["model"] = f"hybrid_{variant}"
        metrics["checkpoint_epoch"] = ckpt["epoch"]
        rows.append(metrics)
        print(f"[hybrid_{variant}] sample_mean_f1={metrics['sample_mean_f1']:.4f}  "
              f"macro_f1={metrics['macro_f1']:.4f}  (epoch {ckpt['epoch']})")
        del model

    df = pd.DataFrame(rows)
    out_path = os.path.join(C.OUTPUT_DIR, f"hybrid_results_fold{args.fold}.csv")
    df.to_csv(out_path, index=False)
    print("\nSaved:", out_path)

    baseline_path = os.path.join(C.OUTPUT_DIR, f"leaderboard_comparable_fold{args.fold}.csv")
    if os.path.exists(baseline_path) and len(df):
        base_df = pd.read_csv(baseline_path).rename(columns={"backbone": "model"})
        combined = pd.concat([base_df[["model", "sample_mean_f1", "macro_f1"]],
                               df[["model", "sample_mean_f1", "macro_f1"]]], ignore_index=True)
        combined = combined.sort_values("sample_mean_f1", ascending=False)
        print("\n=== Step 2 baselines + Step 3 hybrid, side by side ===")
        print(combined.to_string(index=False))
        combined.to_csv(os.path.join(C.OUTPUT_DIR, f"step2_vs_step3_fold{args.fold}.csv"), index=False)
    else:
        print(f"\n(Run src.evaluate_checkpoints first to get {baseline_path} for a side-by-side comparison.)")


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/evaluate_hybrid.py")

wrote src/evaluate_hybrid.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/evaluate_checkpoints.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Recompute metrics (including sample_mean_f1, the actual UNIFESP Kaggle
competition leaderboard metric) on already-trained checkpoints, WITHOUT
retraining. Use this after a train.py run so you don't have to redo 30
epochs just because we added a new metric.

Kaggle usage (after your 8 backbones are already trained):
    python -m src.evaluate_checkpoints --fold 0
"""
import argparse
import json
import os

import pandas as pd
import torch
from torch.utils.data import DataLoader

from . import config as C
from .dataset import UnifespDataset, get_transforms, build_uid_to_path
from .models import BASELINE_BACKBONES, build_model, input_size_for
from .losses import build_loss
from .engine import validate


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--fold", type=int, default=0)
    p.add_argument("--batch_size", type=int, default=32)
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    index_df = pd.read_csv(C.INDEX_CSV)
    folds_df = pd.read_csv(C.FOLDS_CSV)
    train_index = index_df[index_df["split"] == "train"].merge(folds_df[["uid", "fold"]], on="uid", how="inner")
    val_df = train_index[train_index["fold"] == args.fold].reset_index(drop=True)

    print("indexing DICOM files under", C.DATA_ROOT, "...")
    uid_to_path = build_uid_to_path(C.DATA_ROOT)

    label_cols = [f"label_{k}_{C.LABEL_NAMES[k].replace(' ', '_')}" for k in range(C.N_CLASSES)]
    loss_fn = build_loss(None, device=device)  # pos_weight doesn't matter for eval-only loss value

    rows = []
    for name in BASELINE_BACKBONES:
        run_dir = os.path.join(C.OUTPUT_DIR, f"{name}_fold{args.fold}")
        ckpt_path = os.path.join(run_dir, "checkpoint_best.pt")
        if not os.path.exists(ckpt_path):
            print(f"[skip] {name}: no checkpoint_best.pt in {run_dir}")
            continue

        img_size = input_size_for(name)
        val_ds = UnifespDataset(val_df, C.DATA_ROOT, transform=get_transforms(img_size, train=False),
                                 uid_to_path=uid_to_path)
        val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True)

        model = build_model(name, pretrained=False).to(device)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)  # our own checkpoint, trusted
        model.load_state_dict(ckpt["model_state"])

        metrics, _, _ = validate(model, val_loader, loss_fn, device)
        metrics["backbone"] = name
        metrics["checkpoint_epoch"] = ckpt["epoch"]
        rows.append(metrics)
        print(f"[{name}] sample_mean_f1={metrics['sample_mean_f1']:.4f}  "
              f"(macro_f1={metrics['macro_f1']:.4f}, epoch {ckpt['epoch']})")
        del model

    df = pd.DataFrame(rows).sort_values("sample_mean_f1", ascending=False)
    out_path = os.path.join(C.OUTPUT_DIR, f"leaderboard_comparable_fold{args.fold}.csv")
    df.to_csv(out_path, index=False)
    print("\nSaved:", out_path)
    print(df[["backbone", "sample_mean_f1", "macro_f1", "micro_f1", "subset_accuracy"]].to_string(index=False))
    print("\nFor reference, the real UNIFESP Kaggle competition's private leaderboard "
          "(same Mean-F1-Score metric, 115 teams) topped out at 0.93378 (1st place), "
          "with the top ~49 teams all above 0.898.")


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/evaluate_checkpoints.py")

wrote src/evaluate_checkpoints.py


In [ ]:
os.makedirs("/kaggle/working/src", exist_ok=True)
with open("/kaggle/working/src/ensemble_eval.py", "w", encoding="utf-8") as f:
    f.write(r'''"""
Final push: ensemble the best Step 2 baselines + Step 3 hybrids, add
test-time augmentation (horizontal flip), and tune a per-class decision
threshold on the validation fold -- three well-established, NO-RETRAINING
techniques, aimed at closing the small (~0.006) gap to the real UNIFESP
Kaggle competition's private-leaderboard best (0.934, sample_mean_f1).

Reports, in order, how much each lever adds:
    1. best single model (0.5 threshold)              -- our Step 2/3 baseline
    2. + TTA (hflip)                                   -- cheap, per-model
    3. + ensemble (average probs across all 4 models)  -- usually the biggest jump
    4. + per-class threshold tuning                    -- squeezes out the rest

Kaggle usage (after swin_tiny, densenet121, hybrid_plain, hybrid_hierarchical
are all trained in THIS session -- see the notebook):
    python -m src.ensemble_eval --fold 0
"""
import argparse
import os

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from . import config as C
from .dataset import UnifespDataset, get_transforms, build_uid_to_path
from .models import build_model, input_size_for
from .models_hybrid import HybridCNNTransformer, HYBRID_INPUT_SIZE
from .engine import sample_mean_f1
from sklearn.metrics import f1_score

MODEL_SPECS = [
    # (display_name, run_dir_name, loader)
    ("swin_tiny", "swin_tiny_fold{fold}", lambda: build_model("swin_tiny", pretrained=False)),
    ("densenet121", "densenet121_fold{fold}", lambda: build_model("densenet121", pretrained=False)),
    ("hybrid_plain", "hybrid_plain_fold{fold}", lambda: HybridCNNTransformer(pretrained=False)),
    ("hybrid_hierarchical", "hybrid_hierarchical_fold{fold}", lambda: HybridCNNTransformer(pretrained=False)),
]


@torch.no_grad()
def predict_probs(model, loader, device, tta=True):
    """Returns (y_true, y_prob) in the loader's iteration order. With TTA,
    also runs each batch horizontally flipped and averages the two
    sigmoid outputs -- a standard, free accuracy bump at inference time."""
    model.eval()
    all_probs, all_targets = [], []
    for images, targets in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits)
        if tta:
            flipped = torch.flip(images, dims=[3])  # horizontal flip (W axis)
            logits_f = model(flipped)
            probs = (probs + torch.sigmoid(logits_f)) / 2
        all_probs.append(probs.cpu().numpy())
        all_targets.append(targets.numpy())
    return np.concatenate(all_targets), np.concatenate(all_probs)


def tune_per_class_thresholds(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.arange(0.05, 0.96, 0.05)
    n_classes = y_true.shape[1]
    best_t = np.full(n_classes, 0.5)
    for c in range(n_classes):
        best_f1, best_tc = -1.0, 0.5
        for t in grid:
            pred = (y_prob[:, c] >= t).astype(int)
            f1 = f1_score(y_true[:, c], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_tc = f1, t
        best_t[c] = best_tc
    return best_t


def report(name, y_true, y_pred):
    sm_f1 = sample_mean_f1(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"  {name:40s} sample_mean_f1={sm_f1:.4f}  macro_f1={macro:.4f}")
    return sm_f1, macro


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--fold", type=int, default=0)
    p.add_argument("--batch_size", type=int, default=24)
    args = p.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    index_df = pd.read_csv(C.INDEX_CSV)
    folds_df = pd.read_csv(C.FOLDS_CSV)
    train_index = index_df[index_df["split"] == "train"].merge(folds_df[["uid", "fold"]], on="uid", how="inner")
    val_df = train_index[train_index["fold"] == args.fold].reset_index(drop=True)

    print("indexing DICOM files under", C.DATA_ROOT, "...")
    uid_to_path = build_uid_to_path(C.DATA_ROOT)

    y_true = None
    per_model_probs = {}
    rows = []

    for name, run_dir_tmpl, build_fn in MODEL_SPECS:
        run_dir = os.path.join(C.OUTPUT_DIR, run_dir_tmpl.format(fold=args.fold))
        ckpt_path = os.path.join(run_dir, "checkpoint_best.pt")
        if not os.path.exists(ckpt_path):
            print(f"[skip] {name}: no checkpoint_best.pt in {run_dir}")
            continue

        img_size = HYBRID_INPUT_SIZE if "hybrid" in name else input_size_for(name)
        val_ds = UnifespDataset(val_df, C.DATA_ROOT, transform=get_transforms(img_size, train=False),
                                 uid_to_path=uid_to_path)
        val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True)

        model = build_fn().to(device)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state"])

        yt_no_tta, yp_no_tta = predict_probs(model, val_loader, device, tta=False)
        _, yp_tta = predict_probs(model, val_loader, device, tta=True)
        if y_true is None:
            y_true = yt_no_tta
        per_model_probs[name] = yp_tta

        print(f"\n=== {name} ===")
        report("  no TTA, 0.5 threshold   ", yt_no_tta, (yp_no_tta >= 0.5).astype(int))
        report("  + TTA, 0.5 threshold    ", yt_no_tta, (yp_tta >= 0.5).astype(int))
        del model

    if not per_model_probs:
        print("No checkpoints found at all -- nothing to ensemble.")
        return

    print(f"\n=== Ensembling {list(per_model_probs.keys())} (equal-weight average, with TTA) ===")
    ensemble_prob = np.mean(list(per_model_probs.values()), axis=0)
    sm_05, macro_05 = report("ensemble, 0.5 threshold", y_true, (ensemble_prob >= 0.5).astype(int))

    best_thresholds = tune_per_class_thresholds(y_true, ensemble_prob)
    y_pred_tuned = (ensemble_prob >= best_thresholds[None, :]).astype(int)
    sm_tuned, macro_tuned = report("ensemble, tuned thresholds", y_true, y_pred_tuned)

    print("\n=== Summary ===")
    print(f"  sample_mean_f1: 0.5-thresh ensemble = {sm_05:.4f}  ->  tuned-threshold ensemble = {sm_tuned:.4f}")
    print(f"  macro_f1:       0.5-thresh ensemble = {macro_05:.4f}  ->  tuned-threshold ensemble = {macro_tuned:.4f}")
    print("  (for reference: real Kaggle competition 1st place = 0.93378 sample_mean_f1, 115 teams)")

    out = pd.DataFrame([{
        "setup": "ensemble_0.5_threshold", "sample_mean_f1": sm_05, "macro_f1": macro_05,
    }, {
        "setup": "ensemble_tuned_threshold", "sample_mean_f1": sm_tuned, "macro_f1": macro_tuned,
    }])
    out_path = os.path.join(C.OUTPUT_DIR, f"ensemble_results_fold{args.fold}.csv")
    out.to_csv(out_path, index=False)
    print("\nSaved:", out_path)
    np.save(os.path.join(C.OUTPUT_DIR, f"ensemble_thresholds_fold{args.fold}.npy"), best_thresholds)


if __name__ == "__main__":
    main()
'''.lstrip('\n'))
print("wrote src/ensemble_eval.py")

wrote src/ensemble_eval.py


3. Environment setup

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

os.environ["UNIFESP_DATA_ROOT"] = IMAGES_DATASET
os.environ["UNIFESP_INDEX_ROOT"] = "/kaggle/working"
os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"
os.environ["UNIFESP_CACHE_DIR"] = "/kaggle/working/image_cache"

!pip install -q timm albumentations iterative-stratification thop grad-cam

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())

CUDA available: True | GPU count: 2


4. Generate index.csv + the patient-level stratified 5-fold split

In [ ]:
!cd /kaggle/working && python -m src.build_index

Indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
train dcm files: 1738 | test dcm files: 743
Saved index to /kaggle/working/index.csv | rows: 2481

Unique patients -> train: 768, test: 330
Patient overlap between train/test (Kaggle's own split): 0
Train patients with >1 image: 483 / 768


In [ ]:
!cd /kaggle/working && python -m src.make_splits

Patients: 768 | images: 1738
Per-class positive patient count:
 label_0_Abdomen             55
label_1_Ankle               26
label_2_Cervical_Spine      46
label_3_Chest              457
label_4_Clavicles            5
label_5_Elbow               13
label_6_Feet                32
label_7_Finger              16
label_8_Forearm              9
label_9_Hand                42
label_10_Hip                18
label_11_Knee               44
label_12_Lower_Leg          13
label_13_Lumbar_Spine       47
label_14_Others             47
label_15_Pelvis             62
label_16_Shoulder           18
label_17_Sinus              15
label_18_Skull               5
label_19_Thigh              10
label_20_Thoracic_Spine     38
label_21_Wrist              61
dtype: int64
Saved /kaggle/working/splits/folds.csv
Patients appearing in >1 fold (should be 0): 0

Per-fold image count:
 fold
0    364
1    356
2    357
3    338
4    323
Name: count, dtype: int64

Per-fold positive image count per class:
      label_0

5. Pre-cache DICOM -> PNG

In [ ]:
!cd /kaggle/working && python -m src.cache_images

Indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
Found 2481 files. Caching to /kaggle/working/image_cache ...
  250/2481 (22.7s elapsed)
  500/2481 (45.0s elapsed)
  750/2481 (69.1s elapsed)
  1000/2481 (95.1s elapsed)
  1250/2481 (120.4s elapsed)
  1500/2481 (145.2s elapsed)
  1750/2481 (168.1s elapsed)
  2000/2481 (190.8s elapsed)
  2250/2481 (213.8s elapsed)
Done. Cached 2481 images in 237.1s -> /kaggle/working/image_cache


6. Train the 2 baselines (fast, one per GPU)

Swin-Tiny and DenseNet121 were Step 2's two best backbones -- these are small/fast relative to the hybrid, so this pair should finish in well under an hour.

In [ ]:
import subprocess

def launch_baseline(backbone, gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    log = open(f"/kaggle/working/{backbone}.log", "w")
    return subprocess.Popen(
        ["python", "-m", "src.train", "--backbone", backbone, "--fold", "0",
         "--epochs", "30", "--batch_size", "32", "--max_hours", "5"],
        cwd="/kaggle/working", env=env, stdout=log, stderr=subprocess.STDOUT,
    )

baselines = ["swin_tiny", "densenet121"]
procs = list(zip(baselines, [launch_baseline(b, gpu) for gpu, b in enumerate(baselines)]))
for name, p in procs:
    rc = p.wait()
    print(f"[{'ok' if rc == 0 else 'FAILED'}] {name} (exit {rc})" + ("" if rc == 0 else f" -- check /kaggle/working/{name}.log"))

[ok] swin_tiny (exit 0)
[ok] densenet121 (exit 0)


7. Train the 2 hybrid variants (heavier, one per GPU)

--max_hours 10 leaves a buffer in the 12h session; re-run this same cell in a new session if it cuts off -- it resumes from checkpoint_last.pt.

In [ ]:
def launch_hybrid(loss_variant, gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    log = open(f"/kaggle/working/hybrid_{loss_variant}.log", "w")
    return subprocess.Popen(
        ["python", "-m", "src.train_hybrid", "--loss", loss_variant, "--fold", "0",
         "--epochs", "30", "--batch_size", "24", "--max_hours", "10"],
        cwd="/kaggle/working", env=env, stdout=log, stderr=subprocess.STDOUT,
    )

variants = ["plain", "hierarchical"]
procs = list(zip(variants, [launch_hybrid(v, gpu) for gpu, v in enumerate(variants)]))
for name, p in procs:
    rc = p.wait()
    print(f"[{'ok' if rc == 0 else 'FAILED'}] hybrid_{name} (exit {rc})" + ("" if rc == 0 else f" -- check /kaggle/working/hybrid_{name}.log"))

[ok] hybrid_plain (exit 0)
[ok] hybrid_hierarchical (exit 0)


Check progress any time (run in a new cell while training is in progress):

In [ ]:
!tail -n 20 /kaggle/working/swin_tiny.log

epoch 15: train_loss=0.0318 val_loss=0.2009 macro_f1=0.8535 (11.9s)
  -> new best macro_f1=0.8535, saved checkpoint_best.pt
epoch 16: train_loss=0.0249 val_loss=0.1836 macro_f1=0.8653 (11.9s)
  -> new best macro_f1=0.8653, saved checkpoint_best.pt
epoch 17: train_loss=0.0181 val_loss=0.1896 macro_f1=0.8544 (12.1s)
epoch 18: train_loss=0.0166 val_loss=0.2015 macro_f1=0.8541 (12.0s)
epoch 19: train_loss=0.0131 val_loss=0.2063 macro_f1=0.8552 (11.8s)
epoch 20: train_loss=0.0126 val_loss=0.2031 macro_f1=0.8512 (11.8s)
epoch 21: train_loss=0.0111 val_loss=0.2128 macro_f1=0.8476 (11.8s)
epoch 22: train_loss=0.0095 val_loss=0.2117 macro_f1=0.8656 (11.8s)
  -> new best macro_f1=0.8656, saved checkpoint_best.pt
epoch 23: train_loss=0.0081 val_loss=0.2062 macro_f1=0.8622 (11.9s)
epoch 24: train_loss=0.0094 val_loss=0.2099 macro_f1=0.8678 (11.7s)
  -> new best macro_f1=0.8678, saved checkpoint_best.pt
epoch 25: train_loss=0.0095 val_loss=0.2105 macro_f1=0.8664 (11.7s)
epoch 26: train_loss=0.0099 

8. Ensemble + TTA + per-class threshold tuning

Prints each model's solo score, then the ensembled score at both the default 0.5 threshold and with per-class tuned thresholds -- so you can see exactly how much each lever (TTA, ensembling, threshold tuning) contributed.

In [ ]:
!cd /kaggle/working && python -m src.ensemble_eval --fold 0

indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...

=== swin_tiny ===
    no TTA, 0.5 threshold                  sample_mean_f1=0.9299  macro_f1=0.8678
    + TTA, 0.5 threshold                   sample_mean_f1=0.9308  macro_f1=0.8768

=== densenet121 ===
    no TTA, 0.5 threshold                  sample_mean_f1=0.9286  macro_f1=0.8677
    + TTA, 0.5 threshold                   sample_mean_f1=0.9322  macro_f1=0.8691

=== hybrid_plain ===
    no TTA, 0.5 threshold                  sample_mean_f1=0.9321  macro_f1=0.8949
    + TTA, 0.5 threshold                   sample_mean_f1=0.9331  macro_f1=0.8865

=== hybrid_hierarchical ===
    no TTA, 0.5 threshold                  sample_mean_f1=0.9184  macro_f1=0.8679
    + TTA, 0.5 threshold                   sample_mean_f1=0.9166  macro_f1=0.8646

=== Ensembling ['swin_tiny', 'densenet121', 'hybrid_plain', 'hybrid_hierarchical'] (equal-weight average, with TTA) ===
  ensemble, 0.5 threshold 

In [ ]:
import pandas as pd
pd.read_csv("/kaggle/working/runs/ensemble_results_fold0.csv")

	setup	sample_mean_f1	macro_f1
0	ensemble_0.5_threshold	0.935806	0.888044
1	ensemble_tuned_threshold	0.944505	0.941780

9. Before this session ends

Click "Save Version" now, or at minimum download the runs/ folder (all 4 models' checkpoints + the ensemble results CSV).